Just a simple cell to check if everything is well installed 

In [18]:
import sys
import pandas as pd
import numpy as np
import duckdb
import requests
from bs4 import BeautifulSoup
import matplotlib.pyplot as plt
import seaborn as sns
import sklearn
import pyarrow
import openpyxl
import mlxtend
import ucimlrepo

print("Python:", sys.version.split()[0])
print("pandas:", pd.__version__)
print("numpy:", np.__version__)
print("duckdb:", duckdb.__version__)
print("requests:", requests.__version__)
print("beautifulsoup4: OK")
print("matplotlib:", plt.matplotlib.__version__)
print("seaborn:", sns.__version__)
print("scikit-learn:", sklearn.__version__)
print("pyarrow:", pyarrow.__version__)
print("openpyxl:", openpyxl.__version__)
print("mlxtend:", mlxtend.__version__)
print("ucimlrepo: OK")

Python: 3.13.5
pandas: 2.3.3
numpy: 2.5.1
duckdb: 1.5.4
requests: 2.34.2
beautifulsoup4: OK
matplotlib: 3.11.1
seaborn: 0.13.2
scikit-learn: 1.9.0
pyarrow: 25.0.1
openpyxl: 3.1.5
mlxtend: 0.23.4
ucimlrepo: OK


## Acquisition strategy


The acquisition has two stages:

1. `/discover/movie` is used to retrieve movies matching the registered scope:
   - release date: 2020-01-01 to 2025-12-31
   - origin country: US
   - genre: Drama OR Thriller
   - minimum vote count: 100

2. `/movie/{movie_id}/credits` is queried for every selected movie to retrieve:
   - director(s)
   - producer(s)
   - cinematographer(s)
   - top-6 billed actors


In [19]:
import os
import json
import time
import random
import hashlib
from pathlib import Path
from email.utils import parsedate_to_datetime
from datetime import datetime, timezone

import requests
import pandas as pd

from dotenv import load_dotenv

load_dotenv()

URL = "https://api.themoviedb.org/3"
DISCOVER = f"{URL}/discover/movie"

TMDB_TOKEN = os.getenv("TMDB_TOKEN")

assert TMDB_TOKEN, "TMDB_TOKEN is missing from .env"

HEADERS = {
    "Authorization": f"Bearer {TMDB_TOKEN}",
    "Accept": "application/json"
}

CACHE = Path("cache")
CACHE.mkdir(exist_ok=True)


print("TMDB credentials loaded ")
print("Token available:", bool(TMDB_TOKEN))
print("Token length:", len(TMDB_TOKEN))

TMDB credentials loaded 
Token available: True
Token length: 239


In [20]:
PARAMS = {
    "primary_release_date.gte": "2020-01-01",
    "primary_release_date.lte": "2025-12-31",
    "with_origin_country": "US",
    "with_genres": "18|53",
    "vote_count.gte": 100,
    "include_adult": "false",
    "include_video": "false",
    "language": "en-US"
}

def cache_key(url, params=None):
    payload = url + "|" + json.dumps(params or {}, sort_keys=True)
    return hashlib.sha256(payload.encode()).hexdigest()[:16]

def http_get(url, params=None, headers=HEADERS, attempts=5):
    for k in range(attempts):

        try:
            r = requests.get(url,params=params,headers=headers,timeout=10)

        except requests.RequestException as exc:
            status, retry_after = repr(exc), 0

        else:

            if r.status_code == 200:
                return r.json()

            if r.status_code not in (429, 500, 502, 503, 504):
                r.raise_for_status()

            # Temporary error -> retry
            status = r.status_code

            try:
                retry_after = int(r.headers.get("Retry-After", 0))
            except ValueError:
                retry_after = 0

        if k + 1 < attempts:
            wait = retry_after or random.uniform(0.001, 2**k)

            print(
                f"{status} on attempt {k + 1}, "
                f"sleeping {wait:.1f}s"
            )

            time.sleep(wait)

    raise RuntimeError(
        f"gave up after {attempts} attempts: {url}"
    )


def cached_get(url, params=None):
    path = CACHE / f"{cache_key(url, params)}.json"

    timestart = datetime.now(timezone.utc)
    if path.exists():
        return json.loads(path.read_text())

    body = http_get(url, params)
    timeend = datetime.now(timezone.utc)

    path.write_text(
        json.dumps(body)
    )

    return body

In [4]:
## test if it works 
params = PARAMS.copy()
params["page"] = 1



payload = cached_get(
    DISCOVER,
    params
)


print("page:", payload["page"])
print("total pages:", payload["total_pages"])
print("total results:", payload["total_results"])
print("results on page:", len(payload["results"]))


page: 1
total pages: 61
total results: 1206
results on page: 20


In [21]:
##getting all movies from the API

movies = []
page = 1
total_pages = None

timestart = datetime.now(timezone.utc)
while True:

    params = PARAMS.copy()
    params["page"] = page

    payload = cached_get(
        DISCOVER,
        params
    )

    movies += payload["results"]

    if total_pages is None:
        total_pages = payload["total_pages"]

    print(
        f"page {page}/{total_pages} "
        f"- {len(movies)} movies"
    )

    if page >= total_pages:
        break

    page += 1

timeend = datetime.now(timezone.utc)
print("time start:", timestart)
print("time end:", timeend)



page 1/61 - 20 movies
page 2/61 - 40 movies
page 3/61 - 60 movies
page 4/61 - 80 movies
page 5/61 - 100 movies
page 6/61 - 120 movies
page 7/61 - 140 movies
page 8/61 - 160 movies
page 9/61 - 180 movies
page 10/61 - 200 movies
page 11/61 - 220 movies
page 12/61 - 240 movies
page 13/61 - 260 movies
page 14/61 - 280 movies
page 15/61 - 300 movies
page 16/61 - 320 movies
page 17/61 - 340 movies
page 18/61 - 360 movies
page 19/61 - 380 movies
page 20/61 - 400 movies
page 21/61 - 420 movies
page 22/61 - 440 movies
page 23/61 - 460 movies
page 24/61 - 480 movies
page 25/61 - 500 movies
page 26/61 - 520 movies
page 27/61 - 540 movies
page 28/61 - 560 movies
page 29/61 - 580 movies
page 30/61 - 600 movies
page 31/61 - 620 movies
page 32/61 - 640 movies
page 33/61 - 660 movies
page 34/61 - 680 movies
page 35/61 - 700 movies
page 36/61 - 720 movies
page 37/61 - 740 movies
page 38/61 - 760 movies
page 39/61 - 780 movies
page 40/61 - 800 movies
page 41/61 - 820 movies
page 42/61 - 840 movies
page 

In [22]:
##normalize the data 
movie_columns = [
    "id",
    "title",
    "original_title",
    "release_date",
    "genre_ids",
    "original_language",
    "vote_count",
    "vote_average",
    "popularity"
]

movies_df = pd.json_normalize(movies) ## all the distinct movies in the dataset
movies_df = movies_df[movie_columns]
movies_genre = movies_df.explode("genre_ids") ## (movie, genre) per row


movies_genre

,id,title,original_title,release_date,genre_ids,original_language,vote_count,vote_average,popularity
0,755898,War of the Worlds,War of the Worlds,2025-07-29,878,en,1160,4.034,107.0445
0,755898,War of the Worlds,War of the Worlds,2025-07-29,53,en,1160,4.034,107.0445
1,591634,False Positive,False Positive,2021-06-18,27,en,204,5.556,49.4602
1,591634,False Positive,False Positive,2021-06-18,9648,en,204,5.556,49.4602
1,591634,False Positive,False Positive,2021-06-18,53,en,204,5.556,49.4602
...,...,...,...,...,...,...,...,...,...
1205,848278,Jurassic Hunt,Jurassic Hunt,2021-08-26,53,en,221,5.000,1.3992
1205,848278,Jurassic Hunt,Jurassic Hunt,2021-08-26,27,en,221,5.000,1.3992
1206,586032,1,1,2020-09-14,18,en,118,6.700,1.1651
1207,715658,The Scientist,The Scientist,2020-09-21,27,en,111,6.000,1.3260


In [23]:

CREDITS_PARAMS = {
    "language": "en-US"
}

JOBS = {
    "Director": "director",
    "Producer": "producer",
    "Director of Photography": "cinematographer"
}

credits_element= []
timestartC = datetime.now(timezone.utc)
for movie_id in movies_df["id"].unique():

    credits_url = f"{URL}/movie/{movie_id}/credits"

    credits = cached_get(credits_url,CREDITS_PARAMS)

    for person in credits["crew"]:

        if person["job"] in JOBS:

            credits_element.append({
                "movie_id": movie_id,
                "person_id": person["id"],
                "person_name": person["name"],
                "role": JOBS[person["job"]],
                "department": person["department"],
                "job": person["job"],
                "billing_order": None
            })


    top_actors = sorted(credits["cast"],key=lambda actor: actor["order"])[:6]

    for actor in top_actors:

        credits_element.append({
            "movie_id": movie_id,
            "person_id": actor["id"],
            "person_name": actor["name"],
            "role": "actor",
            "department": None,
            "job": None,
            "billing_order": actor["order"]
        })

timeendC = datetime.now(timezone.utc)
print("time start:", timestartC)
print("time end:", timeendC)

time start: 2026-10-05 13:15:26.913891+00:00
time end: 2026-10-05 13:20:27.221091+00:00


In [24]:
credits_df = pd.json_normalize(credits_element)
print("Credits DataFrame shape:", credits_df.shape)

display(credits_df)

Credits DataFrame shape: (14252, 7)


,movie_id,person_id,person_name,role,department,job,billing_order
0,755898,1634181,Rich Lee,director,Directing,Director,NaN
1,755898,23430,Timur Bekmambetov,producer,Production,Producer,NaN
2,755898,964093,Patrick Aiello,producer,Production,Producer,NaN
3,755898,1076356,Christopher Probst,cinematographer,Camera,Director of Photography,NaN
4,755898,9778,Ice Cube,actor,None,None,0.0
...,...,...,...,...,...,...,...
14247,715658,2601839,Kristin Keith,actor,None,None,1.0
14248,715658,2678760,Addison McGarry,actor,None,None,2.0
14249,715658,1380098,J. Alan Nelson,actor,None,None,3.0
14250,715658,2678761,David Dittmeir,actor,None,None,4.0


In [25]:
from pathlib import Path

data = Path("data/raw")
data.mkdir(parents=True, exist_ok=True)

movies_df.to_parquet(data / "movies.parquet",index=False)
credits_df.to_parquet(data / "credits.parquet",index=False)

print("movies.parquet:", movies_df.shape)
print("credits.parquet:", credits_df.shape)

movies.parquet: (1208, 9)
credits.parquet: (14252, 7)


In [26]:
movies_check = pd.read_parquet(data / "movies.parquet")

credits_check = pd.read_parquet(data / "credits.parquet")

print("Movies:", movies_check.shape)
print("Credits:", credits_check.shape)

display(movies_check.head())
display(credits_check.head())

Movies: (1208, 9)
Credits: (14252, 7)


,id,title,original_title,release_date,genre_ids,original_language,vote_count,vote_average,popularity
0,755898,War of the Worlds,War of the Worlds,2025-07-29,"[878, 53]",en,1160,4.034,107.0445
1,591634,False Positive,False Positive,2021-06-18,"[27, 9648, 53]",en,204,5.556,49.4602
2,872585,Oppenheimer,Oppenheimer,2023-07-19,"[18, 36]",en,12616,8.022,54.7098
3,361743,Top Gun: Maverick,Top Gun: Maverick,2022-05-21,"[28, 18]",en,11650,8.158,49.5341
4,1368166,The Housemaid,The Housemaid,2025-12-18,"[9648, 53]",en,3163,7.251,48.9580


,movie_id,person_id,person_name,role,department,job,billing_order
0,755898,1634181,Rich Lee,director,Directing,Director,NaN
1,755898,23430,Timur Bekmambetov,producer,Production,Producer,NaN
2,755898,964093,Patrick Aiello,producer,Production,Producer,NaN
3,755898,1076356,Christopher Probst,cinematographer,Camera,Director of Photography,NaN
4,755898,9778,Ice Cube,actor,None,None,0.0


In [31]:
## logs about what we acquired and how many movies we have in the dataset



total_movies = len(movies_df)
total_credits = len(credits_df)
shape_movies = movies_df.shape
shape_credits = credits_df.shape
movies_df.describe(include="all")
movies_df.info()
movies_df.columns
print(movies_df.dtypes)

print(credits_df.dtypes)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1208 entries, 0 to 1207
Data columns (total 9 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   id                 1208 non-null   int64  
 1   title              1208 non-null   object 
 2   original_title     1208 non-null   object 
 3   release_date       1208 non-null   object 
 4   genre_ids          1208 non-null   object 
 5   original_language  1208 non-null   object 
 6   vote_count         1208 non-null   int64  
 7   vote_average       1208 non-null   float64
 8   popularity         1208 non-null   float64
dtypes: float64(2), int64(2), object(5)
memory usage: 85.1+ KB
id                     int64
title                 object
original_title        object
release_date          object
genre_ids             object
original_language     object
vote_count             int64
vote_average         float64
popularity           float64
dtype: object
movie_id           int64
per

In [ ]:
import json
from pathlib import Path
from datetime import datetime




LOG_PATH = Path("acquisition_log.json")
CACHE_DIR = Path("cache")

MOVIES_PARQUET = Path("data/raw/movies.parquet")
CREDITS_PARQUET = Path("data/raw/credits.parquet")


movie_id_col = "movie_id" if "movie_id" in movies_df.columns else "id"

unique_movies = int(movies_df[movie_id_col].nunique())
unique_people = int(credits_df["person_id"].nunique())



movies_missing = {
    col: int(value)
    for col, value in movies_df.isna().sum().items()
    if value > 0
}

credits_missing = {
    col: int(value)
    for col, value in credits_df.isna().sum().items()
    if value > 0
}



movie_statistics = {
    "vote_count": {
        "min": int(movies_df["vote_count"].min()),
        "max": int(movies_df["vote_count"].max()),
        "mean": round(float(movies_df["vote_count"].mean()), 2)
    },

    "vote_average": {
        "min": round(float(movies_df["vote_average"].min()), 2),
        "max": round(float(movies_df["vote_average"].max()), 2),
        "mean": round(float(movies_df["vote_average"].mean()), 2)
    },

    "popularity": {
        "min": round(float(movies_df["popularity"].min()), 2),
        "max": round(float(movies_df["popularity"].max()), 2),
        "mean": round(float(movies_df["popularity"].mean()), 2)
    }
}




role_counts = {
    role: int(count)
    for role, count
    in credits_df["role"].value_counts().items()
}


movies_by_role = {
    role: int(count)
    for role, count
    in (
        credits_df
        .groupby("role")["movie_id"]
        .nunique()
        .items()
    )
}


# Number of actors kept for each movie
actor_counts = (
    credits_df[
        credits_df["role"] == "actor"
    ]
    .groupby("movie_id")
    .size()
)

movies_with_6_actors = int(
    (actor_counts == 6).sum()
)


failed_movie_ids = globals().get(
    "failed_movie_ids",
    []
)


cache_files = (
    len(list(CACHE_DIR.glob("*.json")))
    if CACHE_DIR.exists()
    else 0
)



log = {

    "source": {
        "name": "TMDB API",
        "base_url": "https://api.themoviedb.org/3",
        "discover_endpoint": "/discover/movie",
        "credits_endpoint": "/movie/{movie_id}/credits"
    },


    "scope": {
        "release_date_start": "2020-01-01",
        "release_date_end": "2025-12-31",
        "origin_country": "US",

        "genres": {
            "Drama": 18,
            "Thriller": 53,
            "operator": "OR"
        },

        "minimum_vote_count": 100,

        "cast_rule": "Top 6 actors by lowest billing order",

        "crew_jobs": [
            "Director",
            "Producer",
            "Director of Photography"
        ]
    },

    "acquisition": {

        "movies": {
            "start_time": timestart.isoformat(),
            "end_time": timeend.isoformat(),
            "duration_seconds": round((timeend - timestart).total_seconds(), 2),
            "movies_acquired": unique_movies
        },

        "credits": {
            "start_time": timestartC.isoformat(),
            "end_time": timeendC.isoformat(),
            "duration_seconds": round((timeendC - timestartC).total_seconds(), 2),

            "requests_attempted": unique_movies,

            "requests_succeeded":
                unique_movies - len(failed_movie_ids),

            "requests_failed":
                len(failed_movie_ids),

            "failed_movie_ids": [
                int(x) for x in failed_movie_ids
            ]
        },

        "log_created_at": datetime.now().isoformat()
    },


    "movies_dataset": {
        "path": str(MOVIES_PARQUET),

        "shape": [
            int(movies_df.shape[0]),
            int(movies_df.shape[1])
        ],

        "columns": list(movies_df.columns),

        "unique_movies": unique_movies,

        "duplicate_rows":
            int(movies_df.astype(str).duplicated().sum()),

        "missing_values": movies_missing,

        "statistics": movie_statistics
    },


    "credits_dataset": {
        "path": str(CREDITS_PARQUET),

        "shape": [
            int(credits_df.shape[0]),
            int(credits_df.shape[1])
        ],

        "columns": list(credits_df.columns),

        "unique_people": unique_people,

        "duplicate_rows":
            int(credits_df.astype(str).duplicated().sum()),

        "missing_values": credits_missing,

        "role_counts": role_counts,

        "movies_by_role": movies_by_role,

        "movies_with_6_actors": movies_with_6_actors
    },


    "cache": {
        "path": str(CACHE_DIR),
        "cached_responses": cache_files
    },

}




with LOG_PATH.open("w", encoding="utf-8") as f:
    json.dump(
        log,
        f,
        indent=2,
        ensure_ascii=False
    )



print("Log saved:", LOG_PATH)

print("\nMovies:")
print("Shape:", movies_df.shape)
print("Unique movies:", unique_movies)

print("\nCredits:")
print("Shape:", credits_df.shape)
print("Unique people:", unique_people)

print("\nRoles:")
display(
    credits_df["role"]
    .value_counts()
    .rename_axis("role")
    .reset_index(name="count")
)

Log saved: acquisition_log.json

Movies:
Shape: (1208, 9)
Unique movies: 1208

Credits:
Shape: (14252, 7)
Unique people: 8839

Roles:


,role,count
0,actor,7191
1,producer,4573
2,director,1289
3,cinematographer,1199
